# HAR–gene regulatory research: Stage 2, verified-data Colab pilot

Prepared 18 November 2026. **Run cells in order**. Includes source verification, raw-file metadata audit, Pal interaction comparisons, optional coordinate overlap (blocked until build/convention verification), optional real-effect classification and statistical benchmarking. **No fabricated gene findings or simulated biological results.**

### Scientific constraints

180 perturbation targets ≠ 180 validated HAR-to-gene links. GSE292632 targeted 106 cis-regulated regions (not necessarily HARs). GSE305245 is postnatal tissue context. Contacts are not proof of gene regulation. Cells and lanes are not biological replicates. Missing or blocked overlap is **NA**, never zero.

Sources: [GSE270828](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE270828), [GSE292632](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE292632), [GSE305245](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE305245), [Pal Dryad](https://doi.org/10.5061/dryad.c59zw3rkj).

In [ ]:
import os, io, gzip, hashlib, json, shutil, re, zipfile, traceback
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import requests, numpy as np, pandas as pd
import matplotlib.pyplot as plt

BASE=Path('/content/har_stage2') if Path('/content').exists() else Path.cwd()/'har_stage2'
DATA=BASE/'data'; OUT=BASE/'outputs'
for p in (DATA,OUT):p.mkdir(parents=True,exist_ok=True)
MAX_FILE_MB=16
USER_AGENT='ResearchSourceAudit/1.0 (academic public data reproducibility; contact via GEO repositories)'
print('Source directory:',DATA)

In [ ]:
# Analysis utilities (same logic as the independently tested har_research_core.py)
"""Conservative, audit-friendly methods for the HAR feasibility study.

The module never labels non-significant perturbations as bona-fide negatives and
never assumes an unknown reference assembly matches GRCh38. All experimental
inferences require data provided by the originating studies.
"""
from __future__ import annotations

import gzip
import hashlib
import math
from pathlib import Path
import re
from typing import Optional

import numpy as np
import pandas as pd

PAL_COLUMNS = ['Bait_chr','Bait_start','Bait_end','Bait_name',
               'OtherEnd_chr','OtherEnd_start','OtherEnd_end','CHiCAGO_score']
REGION_COLUMNS = ['chrom','start','end','region_id']
OVERLAP_COLUMNS = ['screen_region_id','pal_bait_name','chrom','screen_start','screen_end',
                   'bait_start','bait_end','overlap_bp','fraction_screen_covered']


def sha256_file(path: str | Path, block_size: int = 1024*1024) -> str:
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for block in iter(lambda:stream.read(block_size),b''):
            h.update(block)
    return h.hexdigest()


def norm_chrom(x) -> str:
    v=str(x).strip().lower()
    if v.startswith('chr'): v=v[3:]
    if v in ('m','mt','mitochondria'): return 'MT'
    return v.upper()


def read_bed(path: str | Path) -> pd.DataFrame:
    """Read a BED file (0-based half-open) without silently dropping invalid intervals."""
    d=pd.read_csv(path,sep='\t',header=None,comment='#',dtype=str,compression='infer')
    if d.shape[1]<3: raise ValueError('BED must have >=3 columns')
    r=pd.DataFrame({'chrom':d.iloc[:,0].map(norm_chrom),
                    'start':pd.to_numeric(d.iloc[:,1],errors='raise').astype('int64'),
                    'end':pd.to_numeric(d.iloc[:,2],errors='raise').astype('int64'),
                    'region_id':d.iloc[:,3].astype(str) if d.shape[1]>3 else ['region_'+str(i+1) for i in range(len(d))]})
    if ((r.start<0)|(r.end<=r.start)).any():
        raise ValueError('Invalid BED intervals: start<0 or end<=start')
    return r.drop_duplicates().reset_index(drop=True)


def read_pal(path: str | Path) -> pd.DataFrame:
    """Load the Pal et al. Dryad interaction table, preserving all associations."""
    d=pd.read_csv(path,sep='\t',dtype=str)
    if not set(PAL_COLUMNS).issubset(d.columns):
        # Some distribution versions are separated by spaces or commas.
        d=pd.read_csv(path,sep=None,engine='python',dtype=str)
    missing=set(PAL_COLUMNS)-set(d.columns)
    if missing: raise ValueError(f'Pal table missing mandatory columns: {sorted(missing)}; detected: {list(d.columns)}')
    for c in ['Bait_start','Bait_end','OtherEnd_start','OtherEnd_end']:
        d[c]=pd.to_numeric(d[c],errors='raise').astype('int64')
    d['CHiCAGO_score']=pd.to_numeric(d['CHiCAGO_score'],errors='raise')
    for c in ['Bait_chr','OtherEnd_chr']: d[c+'_norm']=d[c].map(norm_chrom)
    for left,right in [('Bait_start','Bait_end'),('OtherEnd_start','OtherEnd_end')]:
        if (d[left]>=d[right]).any(): raise ValueError(f'Invalid Pal coordinate interval {left}/{right}')
    return d


def unique_pal_baits(contacts: pd.DataFrame) -> pd.DataFrame:
    d=contacts[['Bait_chr_norm','Bait_start','Bait_end','Bait_name']].copy()
    d.columns=['chrom','start','end','bait_name']
    return d.drop_duplicates().reset_index(drop=True)


def coordinate_overlap(screen_regions: pd.DataFrame,
                       pal_baits: pd.DataFrame,
                       screen_build: Optional[str],
                       pal_build: Optional[str],
                       screen_interval_convention: str='0-based-half-open',
                       pal_interval_convention: Optional[str]=None) -> pd.DataFrame:
    """Return overlaps only when SAME verified build and interval convention.

    Users must supply Pal interval convention after checking README / author
    coordinates. A null/unknown build must never be treated as zero overlap.
    """
    if not screen_build or not pal_build:
        raise ValueError('BLOCKED: unknown reference genome build')
    if screen_build.lower().replace('hg38','grch38') != pal_build.lower().replace('hg38','grch38'):
        raise ValueError(f'BLOCKED: genome-build mismatch: {screen_build} / {pal_build}')
    if not pal_interval_convention:
        raise ValueError('BLOCKED: Pal genomic interval convention not independently verified')
    if screen_interval_convention != pal_interval_convention:
        raise ValueError('BLOCKED: genomic coordinate conventions are different')
    if not {'chrom','start','end','region_id'}.issubset(screen_regions):
        raise ValueError('Screen regions lack required fields')
    if not {'chrom','start','end','bait_name'}.issubset(pal_baits):
        raise ValueError('Pal baits lack required fields')
    hits=[]
    groups={k:g for k,g in pal_baits.groupby('chrom',sort=False)}
    for r in screen_regions.itertuples(index=False):
        x=groups.get(norm_chrom(r.chrom))
        if x is None:continue
        subset=x[(x.start<r.end)&(x.end>r.start)]
        for bait in subset.itertuples(index=False):
            overlap=max(0,min(r.end,bait.end)-max(r.start,bait.start))
            if overlap>0:
                hits.append((r.region_id,bait.bait_name,norm_chrom(r.chrom),int(r.start),int(r.end),
                             int(bait.start),int(bait.end),overlap,overlap/(r.end-r.start)))
    return pd.DataFrame(hits,columns=OVERLAP_COLUMNS).drop_duplicates()


def parse_geo_soft(path: str | Path) -> pd.DataFrame:
    """GEO GSM is a sample accession, *not* necessarily a biological donor."""
    opener=gzip.open if str(path).endswith('.gz') else open
    rows=[];cur=None
    with opener(path,'rt',encoding='utf-8',errors='replace') as f:
        for line in f:
            line=line.rstrip('\n')
            if line.startswith('^SAMPLE = '):
                if cur is not None:rows.append(cur)
                cur={'gsm':line.split('=',1)[1].strip(),'title':'','characteristics':[],
                     'source_name':'','library_strategy':''}
            elif cur is not None:
                for k,col in [('!Sample_title = ','title'),('!Sample_source_name_ch1 = ','source_name'),
                              ('!Sample_library_strategy = ','library_strategy')]:
                    if line.startswith(k):cur[col]=line[len(k):].strip()
                if line.startswith('!Sample_characteristics_ch1 = '):
                    cur['characteristics'].append(line.split('=',1)[1].strip())
    if cur is not None:rows.append(cur)
    if not rows:return pd.DataFrame(columns=['gsm','title','source_name','characteristics','library_strategy'])
    for r in rows:r['characteristics']=' | '.join(r['characteristics'])
    d=pd.DataFrame(rows)
    if d.gsm.duplicated().any():raise ValueError('Repeated GSM accession in parsed SOFT metadata')
    return d


def classify_perturbation_results(df: pd.DataFrame,
                                  q_threshold: float = .05) -> pd.DataFrame:
    """Classify positive perturbation responses; all other tested pairs remain UNKNOWN.

    Input must be a study-supplied, *tested* HAR-gene contrast table. The
    original statistical testing and correction universe must be documented.
    """
    required={'har_id','gene_id','effect_log2fc','fdr','tested','analysis_context'}
    miss=required-set(df.columns)
    if miss:raise ValueError('Perturbation table missing: '+','.join(sorted(miss)))
    out=df.copy()
    if out[['har_id','gene_id','analysis_context']].isna().any().any():
        raise ValueError('Null gene/HAR/context IDs not supported')
    if out.duplicated(['har_id','gene_id','analysis_context']).any():
        raise ValueError('Duplicate HAR-gene-context contrasts; resolve source rows before analysis')
    out['fdr']=pd.to_numeric(out['fdr'],errors='coerce')
    out['effect_log2fc']=pd.to_numeric(out['effect_log2fc'],errors='coerce')
    if not bool(out.fdr.dropna().between(0,1).all()):
        raise ValueError('FDR values must be in [0,1]')
    tested=out['tested'].astype(str).str.strip().str.lower().map(
        {'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if tested.isna().any():raise ValueError('Tested field must be true/false, not unknown')
    out['tested_bool']=tested.astype(bool)
    if (out.loc[~out.tested_bool,'fdr'].notna()).any():
        raise ValueError('Untested contrasts must not have an FDR reported')
    sig=(out.tested_bool & out.fdr.notna() & out.effect_log2fc.notna() & (out.fdr<q_threshold))
    out['evidence_status']=np.where(sig,'FDR_SUPPORTED_EXPRESSION_RESPONSE','UNRESOLVED_NOT_CONFIRMED_NEGATIVE')
    out['response_supported']=sig
    return out


def positive_recall_at_k(links: pd.DataFrame,
                         positives: pd.DataFrame,
                         score_col: str,
                         k: int) -> pd.DataFrame:
    """Per-locus recall of experimentally *supported positives* only.

    The score must be specified for all prediction candidates; absent contacts
    can be filled with zero only if transparently interpreted as 'not detected'.
    Returns one row per locus. Do not call these precision or AUROC metrics.
    """
    for c in ('har_id','gene_id',score_col):
        if c not in links.columns: raise ValueError('Prediction links missing '+c)
    if not {'har_id','gene_id'}.issubset(positives.columns):
        raise ValueError('Positive table missing har_id/gene_id')
    if k<=0:raise ValueError('k must be >=1')
    p=positives[['har_id','gene_id']].drop_duplicates()
    psets={h:set(g.gene_id.astype(str)) for h,g in p.groupby('har_id')}
    rows=[]
    if links.duplicated(['har_id','gene_id']).any():
        raise ValueError('Duplicate candidate HAR-gene rows')
    for h,known in psets.items():
        candidates=links[links.har_id.astype(str)==str(h)].copy()
        candidates[score_col]=pd.to_numeric(candidates[score_col],errors='coerce')
        candidates=candidates.dropna(subset=[score_col])
        candidates['gene_id']=candidates['gene_id'].astype(str)
        ranked=candidates.sort_values([score_col,'gene_id'],ascending=[False,True]).gene_id.head(k)
        recovered=len(known & set(ranked))
        rows.append({'har_id':h,'known_positive_genes':len(known),'scored_candidate_genes':len(candidates),
                     'recovered_positive_genes':recovered,'positive_recall':recovered/len(known),
                     'k':k,'score_col':score_col})
    return pd.DataFrame(rows,columns=['har_id','known_positive_genes','scored_candidate_genes',
                                      'recovered_positive_genes','positive_recall','k','score_col'])


def locus_bootstrap_difference(a:pd.DataFrame,b:pd.DataFrame,
                               seed:int=20261118,n_boot:int=2000) -> dict:
    """Paired locus-bootstrap for (A minus B) mean locus recall.

    Requires enough loci for an informative interval. Returns descriptive
    estimates only; does not manufacture a p-value from a tiny sample.
    """
    z=a[['har_id','positive_recall']].merge(b[['har_id','positive_recall']],on='har_id',
                                           suffixes=('_A','_B'),validate='one_to_one')
    n=len(z)
    if n<5:
        return {'status':'INSUFFICIENT_INDEPENDENT_LOCI','n_loci':n,
                'observed_mean_difference':float((z.positive_recall_A-z.positive_recall_B).mean()) if n else None,
                'bootstrap_CI95':None}
    vals=(z.positive_recall_A-z.positive_recall_B).to_numpy(dtype=float)
    rng=np.random.default_rng(seed)
    samples=rng.integers(0,n,size=(n_boot,n))
    means=np.mean(vals[samples],axis=1)
    return {'status':'COMPUTED_DESCRIPTIVE_PAIRED_CI','n_loci':n,
            'observed_mean_difference':float(vals.mean()),
            'bootstrap_CI95':[float(x) for x in np.quantile(means,[.025,.975])],
            'seed':seed,'n_boot':n_boot}


def summarize_availability(data_dir:str|Path, filenames:list[str]) -> pd.DataFrame:
    rows=[]
    for name in filenames:
        path=Path(data_dir)/name
        rows.append({'file':name,'exists':path.is_file(),
                     'bytes':path.stat().st_size if path.is_file() else None,
                     'sha256':sha256_file(path) if path.is_file() else None})
    return pd.DataFrame(rows)


def bh_adjust(pvalues):
    """BH correction with transparent NA handling, for newly computed p-values."""
    p=np.asarray(pvalues,dtype=float)
    res=np.full(p.shape,np.nan,dtype=float)
    mask=np.isfinite(p)
    if np.any((p[mask]<0)|(p[mask]>1)):
        raise ValueError('p values must be 0..1')
    ranked=p[mask]
    m=len(ranked)
    if not m:return res
    order=np.argsort(ranked)
    adjusted=np.minimum.accumulate((ranked[order]*m/(np.arange(m)+1))[::-1])[::-1]
    res_indices=np.where(mask)[0]
    res[res_indices[order]]=np.clip(adjusted,0,1)
    return res

In [ ]:
SOURCES={
    'GSE270828_feature': ('GSE270828_feature_README.csv', 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/suppl/GSE270828_feature_README.csv'),
    'GSE270828_SOFT': ('GSE270828_family.soft.gz', 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/soft/GSE270828_family.soft.gz'),
    'GSE292632_regions': ('GSE292632_NPC_4N_CRISPRi_screen_target_regions.bed.gz', 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE292nnn/GSE292632/suppl/GSE292632_NPC_4N_CRISPRi_screen_target_regions.bed.gz'),
    'GSE292632_guide_ref': ('GSE292632_NPC_4N_CRISPRi_screen_gRNA_feature_ref.csv.gz', 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE292nnn/GSE292632/suppl/GSE292632_NPC_4N_CRISPRi_screen_gRNA_feature_ref.csv.gz'),
    'GSE292632_SOFT': ('GSE292632_family.soft.gz', 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE292nnn/GSE292632/soft/GSE292632_family.soft.gz'),
    'GSE305245_SOFT': ('GSE305245_family.soft.gz', 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE305nnn/GSE305245/soft/GSE305245_family.soft.gz'),
    'Pal_hNSC': ('HAR_interactions_hNSCs.txt', 'https://datadryad.org/downloads/file_stream/3922898'),
    'Pal_cNSC': ('HAR_interactions_cNSCs.txt', 'https://datadryad.org/downloads/file_stream/3922897'),
    'Pal_human_neurons': ('HAR_interactions_human_neurons.txt', 'https://datadryad.org/downloads/file_stream/3922896'),
    'Pal_readme': ('Pal_Dryad_README.md', 'https://datadryad.org/downloads/file_stream/3922908'),
}

def download_small(name,url,max_mb=MAX_FILE_MB):
    dest=DATA/name; tmp=DATA/(name+'.part')
    if dest.is_file() and dest.stat().st_size:
        return 'previous_or_user_upload',dest.stat().st_size,sha256_file(dest),''
    try:
        with requests.get(url,headers={'User-Agent':USER_AGENT},timeout=(20,100),stream=True) as response:
            response.raise_for_status()
            size=int(response.headers.get('Content-Length','0') or 0)
            limit=int(max_mb*1024**2)
            if size>limit:raise ValueError('Server content-length larger than cap')
            received=0
            with tmp.open('wb') as stream:
                for chunk in response.iter_content(262144):
                    if not chunk:continue
                    received+=len(chunk)
                    if received>limit:raise ValueError('Download exceeded cap')
                    stream.write(chunk)
            if received==0:raise ValueError('Empty source download')
            with tmp.open('rb') as stream:
                first=stream.read(256).lower().lstrip()
            if first.startswith((b'<html',b'<!doctype',b'{"error"',b'{"message"')):
                raise ValueError('Source delivered a web/error page instead of dataset')
            if name.endswith('.gz'):
                with gzip.open(tmp,'rb') as gz:gz.read(32)
            tmp.replace(dest)
            return 'downloaded',received,sha256_file(dest),''
    except Exception as ex:
        tmp.unlink(missing_ok=True)
        return 'UNAVAILABLE',None,None,str(ex)[:280]

manifest=[]
for key,(file,url) in SOURCES.items():
    status,n,sha,err=download_small(file,url)
    manifest.append({'key':key,'file':file,'url':url,'status':status,'bytes':n,'sha256':sha,'error':err})
    print(f'{key:23s} {status:23s}',str(n or ''),err[:90])
pd.DataFrame(manifest).to_csv(OUT/'01_source_manifest.tsv',sep='\t',index=False)

### Source download troubleshooting

GEO sometimes returns 429/403 or the Dryad direct stream may block automated access. That does **not** imply absent data. Download original files manually from accession/Dryad pages, place into `/content/har_stage2/data/` with exact filenames above, and rerun the download cell. Never edit source tables to make a biological match appear. The 2.2 GB GSE270828 RDS is **not downloaded** here.

In [ ]:
# GEO GSM-level metadata review: GSM/reaction/lane counts are NOT independent lines/donors.
metadata=[]
for gse in ['GSE270828','GSE292632','GSE305245']:
    file=DATA/(gse+'_family.soft.gz')
    if not file.exists():
        print(gse,'unavailable, sample count not established')
        continue
    try:
        x=parse_geo_soft(file);x.insert(0,'gse',gse)
        metadata.append(x)
        print(gse,'GSM records:',len(x))
        print(x[['gsm','title','characteristics']].head(3).to_string(index=False,max_colwidth=90))
    except Exception as ex:print(gse,'ERROR',str(ex)[:200])
if metadata:
    pd.concat(metadata,ignore_index=True).to_csv(OUT/'02_GEO_GSM_metadata.tsv',sep='\t',index=False)
(OUT/'02_replication_warning.txt').write_text('GSM/sample lane counts must not be counted as donor/line replicates. Independent biological lines not verified by this notebook.\n')

In [ ]:
# Core file-level descriptive audit: no causal assumptions or gene-effect estimates.
summary=[]; pal_data={}
for role,name in [('hNSC','HAR_interactions_hNSCs.txt'),('cNSC','HAR_interactions_cNSCs.txt'),('human_neurons','HAR_interactions_human_neurons.txt')]:
    path=DATA/name
    if not path.exists():
        summary.append({'dataset':role,'status':'UNAVAILABLE'});continue
    try:
        df=read_pal(path);pal_data[role]=df
        summary.append({'dataset':role,'status':'PARSED','contact_rows':len(df),
                        'unique_bait_labels':df.Bait_name.nunique(),
                        'unique_bait_fragments':len(unique_pal_baits(df)),
                        'n_score_ge_5':int((df.CHiCAGO_score>=5).sum()),
                        'median_score':float(df.CHiCAGO_score.median())})
        df.head(5).to_csv(OUT/f'03_{role}_contact_schema_preview.tsv',sep='\t',index=False)
    except Exception as ex:
        summary.append({'dataset':role,'status':'PARSE_FAILED','error':str(ex)[:350]})
        print(role,'ERROR:',str(ex)[:350])
pd.DataFrame(summary).to_csv(OUT/'03_Pal_contact_descriptive_summary.tsv',sep='\t',index=False)
print(pd.DataFrame(summary).to_string(index=False))

bed_file=DATA/'GSE292632_NPC_4N_CRISPRi_screen_target_regions.bed.gz'
regions=None
if bed_file.exists():
    try:
        regions=read_bed(bed_file)
        regions.to_csv(OUT/'04_GSE292632_target_regions_as_supplied.tsv',sep='\t',index=False)
        print('Target region BED rows:',len(regions),'unique region IDs:',regions.region_id.nunique())
    except Exception as ex: print('BED validation failed:',str(ex))
for name in ['GSE270828_feature_README.csv','GSE292632_NPC_4N_CRISPRi_screen_gRNA_feature_ref.csv.gz']:
    f=DATA/name
    if not f.exists():continue
    try:
        d=pd.read_csv(f,compression='infer',nrows=30)
        print('\n',name,'column names:',list(d.columns),'preview rows:',len(d))
        d.head(5).to_csv(OUT/('04_'+name.replace('.csv.gz','').replace('.csv','')+'_preview.tsv'),sep='\t',index=False)
    except Exception as ex: print(name,'read error',str(ex)[:180])

In [ ]:
# Paired Pal contact descriptions (same HAR bait label across different cell states,
# not a statement of same functional targets or matched biological replicate).
if 'hNSC' in pal_data and 'human_neurons' in pal_data:
    a=set(pal_data['hNSC'].Bait_name.dropna().astype(str))
    b=set(pal_data['human_neurons'].Bait_name.dropna().astype(str))
    x={'status':'EXACT_BAIT_LABEL_COMPARISON','hNSC_baits':len(a),
       'neuron_baits':len(b),'shared_exact_bait_labels':len(a&b),
       'jaccard_bait_labels':len(a&b)/len(a|b) if a|b else None,
       'important':'Not a cell-state difference in gene function; differential ascertainment may contribute.'}
    (OUT/'05_Pal_exact_bait_label_overlap.json').write_text(json.dumps(x,indent=2))
    print(x)
else:print('Pal human NSC/neuron comparative summary skipped: source unavailable')

### Genomic coordinates — mandatory safeguards

We deliberately do **not** assume the GSE292632 target BED and Pal bait fragments are directly comparable merely because another part of the study aligned RNA reads to hg38. BED assembly and the interval conventions must each be confirmed from original study files and methods.

Keep the default settings `None` to block an unverified coordinate join. Once you have independently confirmed both settings, enter them below and rerun. A blocked overlap is not evidence of zero overlap.

In [ ]:
# Only change these two after independently verifying GSE292632 BED build and Pal bait coordinate convention.
VERIFIED_GSE292632_BED_BUILD=None # e.g., 'GRCh38' ONLY IF VERIFIED
VERIFIED_PAL_INTERVAL_CONVENTION=None # '0-based-half-open' ONLY IF VERIFIED
report={'status':'NOT_RUN','screen_build':VERIFIED_GSE292632_BED_BUILD,
        'pal_build':'GRCh38','pal_interval_convention':VERIFIED_PAL_INTERVAL_CONVENTION}
if regions is not None and 'hNSC' in pal_data:
    try:
        matches=coordinate_overlap(regions,unique_pal_baits(pal_data['hNSC']),
             VERIFIED_GSE292632_BED_BUILD,'GRCh38',
             pal_interval_convention=VERIFIED_PAL_INTERVAL_CONVENTION)
        matches.to_csv(OUT/'06_GSE292632_Pal_overlap_CONDITIONAL.tsv',sep='\t',index=False)
        report.update(status='VALID_SAME_BUILD_OVERLAP',overlap_pairs=len(matches),
                      overlapping_screen_regions=matches.screen_region_id.nunique(),
                      overlapping_pal_bait_labels=matches.pal_bait_name.nunique(),
                      note='Geomic intersection only; same regulatory element/functional effect not demonstrated')
    except Exception as ex:report.update(status='BLOCKED',reason=str(ex))
else:report.update(status='SOURCE_UNAVAILABLE',reason='BED and/or Pal hNSC source is missing')
(OUT/'06_coordinate_overlap_status.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))

### Optional Phase 2 — real perturbation effect sizes (only if source table exists)

For a valid benchmark, a **tested HAR–gene contrast table** is required. A guide-feature README, raw gene-count matrix or arbitrary user-provided gene list is not sufficient. Do not relabel missing significance as a negative. You may place a table named `real_perturbation_effects.tsv` in `/content/har_stage2/data/` only after matching the authors' supplementary tests and correction scheme. Columns: `har_id`, `gene_id`, `effect_log2fc`, `fdr`, `tested`, `analysis_context`. The code refuses unsupported schemas.

For positive-recall comparison, create `candidate_links.tsv` in the same folder, containing independently assigned candidates with `har_id`, `gene_id`, `distance_bp`, and `contact_score`. This mapping requires the verified same-build gene annotation and original feature crosswalk; **do not manually guess contact scores**. In the absence of real files, this section exits without output statistics.

In [ ]:
effect_file=DATA/'real_perturbation_effects.tsv'
links_file=DATA/'candidate_links.tsv'
optional_status={'effect_table':'unavailable','prediction_links':'unavailable','benchmarks':'NOT_RUN'}
if effect_file.exists():
    try:
        effects=pd.read_csv(effect_file,sep='\t',dtype={'har_id':str,'gene_id':str})
        effects=classify_perturbation_results(effects)
        effects.to_csv(OUT/'07_actual_perturbation_classification.tsv',sep='\t',index=False)
        optional_status['effect_table']='VALIDATED_SCHEMA'
        optional_status['supported_positive_contrasts']=int(effects.response_supported.sum())
        optional_status['tested_contrasts']=int(effects.tested_bool.sum())
        positives=effects[effects.response_supported][['har_id','gene_id']].drop_duplicates()
        if links_file.exists() and len(positives):
            links=pd.read_csv(links_file,sep='\t',dtype={'har_id':str,'gene_id':str})
            required={'har_id','gene_id','distance_bp','contact_score'}
            if not required.issubset(links):raise ValueError('Candidate links missing '+str(required-set(links)))
            links['distance_rank_score']=-pd.to_numeric(links.distance_bp,errors='raise').abs()
            # This is a missing contact indicator, not a claim of zero biological association.
            links['contact_score']=pd.to_numeric(links.contact_score,errors='coerce').fillna(0.)
            allrows=[];comp=[]
            for k in (1,3,5):
                rA=positive_recall_at_k(links,positives,'contact_score',k)
                rB=positive_recall_at_k(links,positives,'distance_rank_score',k)
                allrows.extend([rA,rB])
                out=locus_bootstrap_difference(rA,rB)
                out.update(k=k,method_A='contact_score',method_B='distance_rank_score')
                comp.append(out)
            pd.concat(allrows,ignore_index=True).to_csv(OUT/'08_positive_recall_per_HAR.tsv',sep='\t',index=False)
            pd.DataFrame(comp).to_csv(OUT/'08_locus_bootstrap_comparison.tsv',sep='\t',index=False)
            optional_status['prediction_links']='PRESENT'
            optional_status['benchmarks']='POSITIVE_ONLY_RECALL_COMPUTED'
    except Exception as ex:
        optional_status['benchmarks']='BLOCKED_INVALID_INPUT'
        optional_status['error']=str(ex)[:550]
(OUT/'07_effect_table_status.json').write_text(json.dumps(optional_status,indent=2))
print(json.dumps(optional_status,indent=2))

In [ ]:
# Generate only descriptive figures supported by real downloaded tables.
if pal_data:
    labels=list(pal_data)
    bait_counts=[pal_data[k].Bait_name.nunique() for k in labels]
    fig,ax=plt.subplots(figsize=(7.5,4.3))
    ax.bar(labels,bait_counts,color='0.4')
    ax.set_ylabel('Unique bait labels in downloaded contact table')
    ax.set_xlabel('Cell state')
    ax.set_title('Descriptive source coverage (not a regulatory effect)')
    ax.spines['top'].set_visible(False);ax.spines['right'].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUT/'09_Pal_bait_coverage_descriptive.png',dpi=220,facecolor='white')
    plt.show();plt.close(fig)
else:
    print('No empirical plot generated: contact source files were unavailable')

In [ ]:
# Self-contained feasibility report, file checksums, version inventory and ZIP export.
import platform
outfiles=sorted([p for p in OUT.glob('*') if p.is_file()])
status={
  'UTC_run_date':datetime.now(timezone.utc).isoformat(),
  'python_version':platform.python_version(),
  'pandas_version':pd.__version__,
  'numpy_version':np.__version__,
  'number_source_files_local':sum(1 for r in SOURCES.values() if (DATA/r[0]).is_file()),
  'source_downloads':{r['key']:r['status'] for r in manifest},
  'coordinate_join':report,
  'perturbation_effects':optional_status,
  'prohibition':'No measured effects or valid absence were inferred from missing data.',
}
(OUT/'10_FINAL_AUDIT.json').write_text(json.dumps(status,indent=2))
(OUT/'10_report.md').write_text("""# Stage 2 audit results

This run documents source acquisition and data feasibility. It does not establish HAR-to-gene causality, evolutionary positive selection, or a neuronal maturation phenotype.

- Source acquisition: see 01_source_manifest.tsv, SHA256 hashes.
- GSM metadata: see 02_GEO_GSM_metadata.tsv; **no independent donor count inferred**.
- Contacts: see 03_Pal_contact_descriptive_summary.tsv, if originals were obtained.
- Genomic join: see 06_coordinate_overlap_status.json; blocked unless independently verified.
- Perturbation effects: see 07_effect_table_status.json; real effects not inferred from guide metadata.
- All findings are provisional pending biological replication and provenance review.
""")
zip_path=BASE/'HAR_Stage2_Run_Outputs.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as arc:
    for p in OUT.glob('*'):
        if p.is_file():arc.write(p,'results/'+p.name)
    arc.writestr('run_info.json',json.dumps(status,indent=2))
print('Audit output:',zip_path)
print('Files:',[p.name for p in OUT.glob('*')])
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Not running in Colab; output ZIP saved locally.')